# Explainability and heterogeneity

Run from the repository root, with the raw OULAD files in `raw/` (see README).

In [ ]:
import os
from pathlib import Path

root = Path.cwd()
for p in [root, *root.parents]:
    if (p / "raw").is_dir():
        root = p
        break
BASE = str(root)

In [3]:
!pip install -q econml shap

import numpy as np, pandas as pd, joblib

SEED = 42
CANONICAL_ATE = 0.004786482048864249

cf_canonical = joblib.load(f'{BASE}/cf_canonical_model.pkl')
full = pd.read_csv(f'{BASE}/oulad_full_corpus.csv')

CONFOUNDERS_V2 = ['region', 'imd_band', 'highest_education', 'age_band', 'gender',
                  'disability', 'code_presentation', 'num_of_prev_attempts',
                  'studied_credits', 'code_module']
CATEGORICAL_V2 = [c for c in CONFOUNDERS_V2
                  if c not in ('num_of_prev_attempts', 'studied_credits')]

n_missing = int(full['imd_band'].isna().sum())
full['imd_band'] = full['imd_band'].fillna('Missing')
print(f'imd_band NaN -> "Missing": {n_missing}   (expect 830)')

X_encoded_v2 = pd.get_dummies(full[CONFOUNDERS_V2],
                              columns=CATEGORICAL_V2, drop_first=True).astype(float)
assert X_encoded_v2.shape == (17529, 40), f'shape drift: {X_encoded_v2.shape}'
assert not X_encoded_v2.isna().any().any(), 'NaN in confounder matrix'

ate_check = float(cf_canonical.ate(X_encoded_v2))
print(f'ATE on rebuilt matrix : {ate_check!r}')
print(f'canonical             : {CANONICAL_ATE!r}')
assert abs(ate_check - CANONICAL_ATE) < 1e-6, 'REBUILD MISMATCH — stop'
print('verified to 1e-6\n')

full['cate'] = cf_canonical.effect(X_encoded_v2)
print(f"CATE  mean={full['cate'].mean()!r}")
print(f"      sd  ={full['cate'].std()!r}")

sample_idx = full.sample(n=500, random_state=SEED).index
X_shap_sample = X_encoded_v2.loc[sample_idx]
shap_values = cf_canonical.shap_values(X_shap_sample, background_samples=500)
shap_explanation = shap_values['Y0']['T0']
print(f'\nSHAP shape: {shap_explanation.values.shape}   (expect (500, 40))')

mean_abs_shap = np.abs(shap_explanation.values).mean(axis=0)
shap_ranking_v2 = (pd.DataFrame({'feature': X_shap_sample.columns,
                                 'mean_abs_shap': mean_abs_shap})
                   .sort_values('mean_abs_shap', ascending=False))
shap_ranking_v2.to_csv(f'{BASE}/shap_ranking_v2.csv', index=False)
print(shap_ranking_v2.head(12).to_string(index=False))

cf, X, Xs = cf_canonical, X_encoded_v2, X_shap_sample
sv = shap_explanation.values
print('\nsetup complete.')

 99%|===================| 497/500 [02:27<00:00]       


SHAP shape: (500, 40)   (expect (500, 40))
                             feature  mean_abs_shap
                     code_module_BBB       0.001824
             code_presentation_2013J       0.001090
                     code_module_CCC       0.000923
                     code_module_DDD       0.000812
                      age_band_35-55       0.000669
             code_presentation_2014B       0.000625
                     code_module_FFF       0.000616
                     studied_credits       0.000539
highest_education_Lower Than A Level       0.000406
             code_presentation_2014J       0.000365
                            gender_M       0.000245
  highest_education_HE Qualification       0.000218

setup complete.


In [ ]:
import sys
import numpy as np
import pandas as pd
import joblib

print("python:", sys.version.split()[0], "| pandas:", pd.__version__, "| numpy:", np.__version__)

MODEL_PATH  = f'{BASE}/cf_canonical_model.pkl'
CORPUS_PATH = f'{BASE}/oulad_full_corpus.csv'
SHAP_SAMPLE_N = 500
RANDOM_SEED   = 42
CANONICAL_ATE = 0.004786

CONFOUNDERS_V2 = ['region', 'imd_band', 'highest_education', 'age_band', 'gender',
                  'disability', 'code_presentation', 'num_of_prev_attempts',
                  'studied_credits', 'code_module']
CATEGORICAL_V2 = ['region', 'imd_band', 'highest_education', 'age_band', 'gender',
                  'disability', 'code_presentation', 'code_module']

cf = cf_canonical
X  = X_encoded_v2
full['imd_band'] = full['imd_band'].fillna('Missing')

X = pd.get_dummies(full[CONFOUNDERS_V2], columns=CATEGORICAL_V2, drop_first=True)
X = X.astype(float)
print(f"corpus n = {len(full):,} | confounder matrix = {X.shape}")

ate_check = cf.ate(X)
print(f"ATE check: {ate_check:.6f}  (canonical = {CANONICAL_ATE})")
assert abs(ate_check - CANONICAL_ATE) < 5e-4, \
    "ATE does not match canonical. Wrong model, wrong corpus, or wrong column order. STOP."

full['cate'] = cf.effect(X)
print(f"CATE: mean={full['cate'].mean():.6f}  sd={full['cate'].std():.6f}  "
      f"min={full['cate'].min():.6f}  max={full['cate'].max():.6f}")
print(f"share of students with negative estimated effect: {(full['cate'] < 0).mean():.1%}")

print("PART A — DIRECTION OF EVERY SUBGROUP EFFECT")

EDU_ORDER = ['No Formal quals', 'Lower Than A Level', 'A Level or Equivalent',
             'HE Qualification', 'Post Graduate Qualification']

out = []
for c in CONFOUNDERS_V2:
    if c == 'num_of_prev_attempts':

        q = pd.cut(full[c], bins=[-0.5, 0.5, np.inf], labels=['0 attempts', '1+ attempts'])
        g = full.groupby(q, observed=True)['cate'].agg(['mean', 'std', 'count'])

    elif c == 'studied_credits':
        q = pd.qcut(full[c], 4, duplicates='drop')
        g = full.groupby(q, observed=True)['cate'].agg(['mean', 'std', 'count'])
    else:
        g = full.groupby(c, observed=True)['cate'].agg(['mean', 'std', 'count'])
        if c == 'highest_education':
            g = g.reindex([e for e in EDU_ORDER if e in g.index])
    print(f"\n--- {c} ---")
    g2 = g.copy()
    g2['vs_ATE'] = g2['mean'] / CANONICAL_ATE
    print(g2.to_string(float_format=lambda v: f"{v:.6f}"))
    for lv, r in g.iterrows():
        out.append({'variable': c, 'level': str(lv), 'mean_cate': r['mean'],
                    'sd': r['std'], 'n': int(r['count']),
                    'ratio_to_ATE': r['mean'] / CANONICAL_ATE})

sub = pd.DataFrame(out)
sub.to_csv('subgroup_directions.csv', index=False)

edu = sub[sub['variable'] == 'highest_education'].set_index('level')
edu = edu.reindex([e for e in EDU_ORDER if e in edu.index])
print(edu[['mean_cate', 'n', 'ratio_to_ATE']].to_string(float_format=lambda v: f"{v:.6f}"))

low = [e for e in ['Lower Than A Level', 'No Formal quals'] if e in edu.index]
high = [e for e in ['HE Qualification', 'Post Graduate Qualification'] if e in edu.index]
if low and high:
    lo, hi = edu.loc[low, 'mean_cate'].mean(), edu.loc[high, 'mean_cate'].mean()
    print("\nMARGINAL, pooled across modules -- DO NOT REPORT THIS RATIO")
    print(f"  low prior education  = {lo!r}")
    print(f"  high prior education = {hi!r}")

eff = full[full['code_module'] != 'CCC']
cell = (eff.groupby(['code_module', 'highest_education'], observed=True)['cate']
        .agg(['mean', 'count']).reset_index())
cell['_wm'] = cell['mean'] * cell['count']
grad = (cell.groupby('highest_education', observed=True)
        .agg(_num=('_wm', 'sum'), n=('count', 'sum'),
             n_modules=('code_module', 'nunique')))
grad['n_weighted_cate'] = grad['_num'] / grad['n']
grad = (grad[['n_weighted_cate', 'n', 'n_modules']]
        .reindex([e for e in EDU_ORDER if e in set(cell['highest_education'])]))
print(grad.to_string(float_format=lambda v: f"{v:.9f}"))

vals = grad['n_weighted_cate'].dropna()
print(f"\nmonotonic decreasing: {bool((vals.diff().dropna() < 0).all())}")

LTA, PG = 'Lower Than A Level', 'Post Graduate Qualification'
if LTA in grad.index and PG in grad.index:
    pass

holds = []
for m, g_ in cell.groupby('code_module', observed=True):
    s_ = g_.set_index('highest_education')['mean']
    if LTA in s_.index and PG in s_.index:
        holds.append((m, bool(s_[LTA] > s_[PG])))
for m, h in holds:
    print(f"    {m}: {'holds' if h else 'REVERSES'}")

print("\n==> COMPENSATORY PATTERN -- least prior education gains MOST.")
print("    R^2 0.0011 on T, 0.0029 on Y). Effect modification is a different")
print("    quantity and this does not weaken the result. Do not conflate them.")
print("    -0.000544, contradicting the subgroup direction. SHAP splits attribution")
print("    across correlated one-hot dummies. Subgroup tables are the primary")
print("    by 12 percentage points (33.6% vs 45.7% of those excluded). The finding")

print("PART B — SIGNED SHAP (conditional on dummy == 1)")

Xs = X_shap_sample
sv = shap_explanation.values
print(f"SHAP array: {sv.shape}")
assert sv.shape[1] == X.shape[1], "SHAP width != confounder width. STOP."

feat = list(X.columns)
rows = []
for j, name in enumerate(feat):
    col, s = Xs.iloc[:, j].values, sv[:, j]
    is_dummy = set(np.unique(col)).issubset({0.0, 1.0})
    if is_dummy and col.sum() > 0:

        signed = s[col == 1].mean()
        base = col.mean()
        note = f"n={int(col.sum())} ({base:.1%} of sample)"
    else:

        signed = np.corrcoef(col, s)[0, 1] * np.abs(s).mean() if col.std() > 0 else 0.0
        note = "continuous (corr-weighted)"
    rows.append({'feature': name,
                 'mean_abs_shap': np.abs(s).mean(),
                 'signed_shap_when_present': signed,
                 'direction': 'RAISES effect' if signed > 0 else 'LOWERS effect',
                 'pct_of_ATE': signed / CANONICAL_ATE,
                 'note': note,
                 'unconditional_mean_BROKEN': s.mean()})

shp = pd.DataFrame(rows).sort_values('mean_abs_shap', ascending=False)
print("\n" + shp.drop(columns=['unconditional_mean_BROKEN'])
      .to_string(index=False, float_format=lambda v: f"{v:.6f}"))
shp.to_csv('shap_signed_v2.csv', index=False)

print("\nProof that the exemplar metric is broken:")
print(f"  max |unconditional mean signed SHAP| across all {len(feat)} features = "
      f"{shp['unconditional_mean_BROKEN'].abs().max():.2e}")
print("  Effectively zero everywhere -- which is why Script 11 would have produced")
print("  directions determined by floating-point noise.")

print("PART C — ATTRIBUTION BY FEATURE FAMILY (all features, not the top 11)")

def family(n):
    if n.startswith('code_module_'):       return 'Course identity'
    if n.startswith('code_presentation_'): return 'Presentation / term'
    return 'Student attribute'

shp['family'] = shp['feature'].map(family)
fam = shp.groupby('family')['mean_abs_shap'].agg(['sum', 'count'])
fam['share'] = fam['sum'] / fam['sum'].sum()
print("\n" + fam.sort_values('sum', ascending=False)
      .to_string(float_format=lambda v: f"{v:.6f}"))

struct = fam.loc[[f for f in ['Course identity', 'Presentation / term'] if f in fam.index],
                 'share'].sum()
print(f"\nCourse-structural share (module + presentation): {struct:.1%}")
print(f"Student-attribute share:                         {1-struct:.1%}")
print(f"\nComputed across all {len(feat)} features. The report's ~75.8% was the share")
print("within the reported top 11 only. Report THIS number, whichever way it lands.")
print("This block previously held that substantial mass at BOTH module and")
print("presentation level required a theory with constructs at each. That is what")
print("       CCC is removed. CCC is 22.8% of 2014B, 20.1% of 2014J and absent from")
print("       2013, so the delivery-level trend was module composition.")
print("       code_module's 0.353 / 0.086. Near-null by an independent method.")
print("Narrow the theory search to COURSE-DESIGN-LEVEL constructs.")

python: 3.13.15 | pandas: 2.2.3 | numpy: 2.1.3
corpus n = 17,529 | confounder matrix = (17529, 40)
ATE check: 0.004786  (canonical = 0.004786)
CATE: mean=0.004786  sd=0.002129  min=-0.001395  max=0.008431
share of students with negative estimated effect: 4.5%

PART A — DIRECTION OF EVERY SUBGROUP EFFECT

--- region ---
                         mean      std  count   vs_ATE
region                                                
East Anglian Region  0.004783 0.002138   1780 0.999468
East Midlands Region 0.005170 0.002096   1189 1.080214
Ireland              0.004893 0.002127    815 1.022330
London Region        0.004480 0.002118   1564 0.935969
North Region         0.004463 0.002055   1049 0.932546
North Western Region 0.005264 0.002161   1386 1.099851
Scotland             0.004208 0.002241   2095 0.879335
South East Region    0.004780 0.001950   1143 0.998742
South Region         0.004653 0.002161   1735 0.972165
South West Region    0.004979 0.002101   1303 1.040319
Wales              

In [ ]:
by_mod = full.groupby('code_module', observed=True)['cate'].agg(['mean', 'std', 'count'])
print(by_mod.to_string(float_format=lambda v: f"{v:.9f}"))

ccc = by_mod.loc['CCC']
others = by_mod.drop('CCC')
print(f"\nCCC      : {ccc['mean']!r}  (sd {ccc['std']!r}, n={int(ccc['count']):,})")
print(f"others   : {others['mean'].min()!r} to {others['mean'].max()!r}")
print(f"ratio    : CCC is {ccc['mean'] / others['mean'].min():.4f}x the smallest other module")
assert abs(ccc['mean'] - 0.000129) < 5e-5, f"CCC CATE drift: {ccc['mean']!r}"

print("Flat across ALL education levels — not an equity story in reverse")
ccc_edu = (full[full['code_module'] == 'CCC']
           .groupby('highest_education', observed=True)['cate']
           .agg(['mean', 'count'])
           .reindex([e for e in EDU_ORDER if e in
                     set(full[full['code_module'] == 'CCC']['highest_education'])]))
print(ccc_edu.to_string(float_format=lambda v: f"{v:.9f}"))
print(f"\nrange across levels: {ccc_edu['mean'].min()!r} to {ccc_edu['mean'].max()!r}")

print("Replicates across BOTH cohorts — the most stable estimate in the study")
coh = (full.groupby(['code_module', 'code_presentation'], observed=True)['cate']
       .agg(['mean', 'count']))
ccc_coh = coh.loc['CCC']
print(ccc_coh.to_string(float_format=lambda v: f"{v:.9f}"))
ccc_drift = float(ccc_coh['mean'].max() - ccc_coh['mean'].min())

drifts = {}
for m in others.index:
    if m in coh.index.get_level_values(0):
        v = coh.loc[m, 'mean']
        if len(v) > 1:
            drifts[m] = float(v.max() - v.min())
mean_other_drift = float(np.mean(list(drifts.values()))) if drifts else float('nan')

print(f"\nCCC cross-cohort drift        : {ccc_drift!r}")
print(f"mean drift, other modules     : {mean_other_drift!r}")
print(f"CCC is {mean_other_drift / ccc_drift:.1f}x more stable")

print("The measurement explanation FAILS")
eng = full.groupby('code_module', observed=True)['oucontent_clicks'].describe()
eng['CV'] = eng['std'] / eng['mean']
print(eng[['count', 'mean', '50%', 'max', 'CV']].to_string(
    float_format=lambda v: f"{v:.3f}"))
print("\nBBB has a LOWER median click count than CCC and a full effect:")
print(f"  BBB median = {eng.loc['BBB', '50%']!r}   CATE = {by_mod.loc['BBB', 'mean']!r}")
print(f"  CCC median = {eng.loc['CCC', '50%']!r}   CATE = {by_mod.loc['CCC', 'mean']!r}")
print("So CCC's null is not 'too little engagement to measure'.")
print("\nF14 adds: CCC ranks 3rd of 40 on partial confounding strength (0.0983),")
print("ordinary among the modules. The null is not a confounder-structure artefact.")

print("What distinguishes CCC — descriptive, no causal claim")
for c in ['code_presentation', 'age_band', 'gender', 'studied_credits',
          'highest_education']:
    print(f"\n--- {c} ---")
    print(pd.crosstab(full['code_module'], full[c], normalize='index').round(3))

pg = full[full['highest_education'] == 'Post Graduate Qualification']
print("That concentration is exactly why the marginal education gradient inflates")

def pres_gap(df, label):
    g = df.groupby('code_presentation', observed=True)['cate'].agg(['mean', 'count'])
    y2013 = [p for p in g.index if p.startswith('2013')]
    y2014 = [p for p in g.index if p.startswith('2014')]
    w13 = np.average(g.loc[y2013, 'mean'], weights=g.loc[y2013, 'count'])
    w14 = np.average(g.loc[y2014, 'mean'], weights=g.loc[y2014, 'count'])
    print(f"\n{label}")
    print(g.to_string(float_format=lambda v: f"{v:.9f}"))
    print(f"  2013 (n-weighted) = {w13!r}")
    print(f"  2014 (n-weighted) = {w14!r}")
    print(f"  gap 2014 - 2013   = {w14 - w13!r}")
    return w14 - w13

gap_all = pres_gap(full, "ALL MODULES  (expected near +0.001067)")
gap_noc = pres_gap(full[full['code_module'] != 'CCC'],
                   "CCC REMOVED  (expected to vanish, near -0.000092)")

print(f"gap with CCC    : {gap_all!r}")
print(f"gap without CCC : {gap_noc!r}")
print(f"share of the gap attributable to CCC composition: "
      f"{100 * (1 - gap_noc / gap_all):.1f}%")

for p in ['2014B', '2014J']:
    sub_p = full[full['code_presentation'] == p]
    print(f"  CCC share of {p}: {(sub_p['code_module'] == 'CCC').mean():.1%}")
print("  CCC share of any 2013 presentation: 0.0% — CCC is absent from 2013.")

print("\nCONSEQUENCE: code_presentation's 22.3% SHAP mass adds little beyond module")
print("theory requirement narrows to course-design-level constructs.")

out_ccc = by_mod.copy()
out_ccc.to_csv(f'{BASE}/ccc_module_cate_v2.csv')
print(f"\nSaved ccc_module_cate_v2.csv")

                   mean         std  count
code_module                               
AAA         0.004476148 0.001067446    612
BBB         0.005054130 0.001081890   4279
CCC         0.000129488 0.000499702   2114
DDD         0.004694001 0.001085256   3608
EEE         0.005500499 0.001125297   1986
FFF         0.006369683 0.001180231   4930

CCC      : np.float64(0.00012948790139526208)  (sd np.float64(0.0004997019197116869), n=2,114)
others   : 0.004476148130755066 to 0.006369682563281485
ratio    : CCC is 0.0289x the smallest other module

------------------------------------------------------------------------------
Flat across ALL education levels — not an equity story in reverse
------------------------------------------------------------------------------
                                    mean  count
highest_education                              
No Formal quals              0.000342917     16
Lower Than A Level           0.000378018    488
A Level or Equivalent        0.0000

In [ ]:
import numpy as np, pandas as pd

EDU_ORDER = ['No Formal quals', 'Lower Than A Level', 'A Level or Equivalent',
             'HE Qualification', 'Post Graduate Qualification']
LTA, PG = 'Lower Than A Level', 'Post Graduate Qualification'

def pres_gaps(df, label):
    g = df.groupby('code_presentation', observed=True)['cate'].agg(['mean', 'count'])
    y13 = [p for p in g.index if p.startswith('2013')]
    y14 = [p for p in g.index if p.startswith('2014')]
    u13, u14 = g.loc[y13, 'mean'].mean(), g.loc[y14, 'mean'].mean()
    w13 = np.average(g.loc[y13, 'mean'], weights=g.loc[y13, 'count'])
    w14 = np.average(g.loc[y14, 'mean'], weights=g.loc[y14, 'count'])
    print(f"\n{label}")
    print(g.to_string(float_format=lambda v: f"{v:.9f}"))
    print(f"  n-weighted   2014 - 2013 = {w14 - w13!r}")
    return u13 - u14, w14 - w13

u_all, w_all = pres_gaps(full, "ALL MODULES")
u_noc, w_noc = pres_gaps(full[full['code_module'] != 'CCC'], "CCC REMOVED")

print(f"reproduced   : {u_all!r} with CCC, {u_noc!r} without")
assert abs(u_all - 0.001067) < 1e-6, f"gap drift: {u_all!r}"
assert abs(u_noc - (-0.000092)) < 1e-6, f"CCC-removed gap drift: {u_noc!r}"
print("both match the recorded values to 1e-6.")
print(f"\nthe gap collapses to {abs(u_noc / u_all):.1%} of its original size once CCC is")
print("removed, and reverses sign -- there is no year trend, only module composition.")
print("Without CCC, 2014B is the HIGHEST presentation of the four.")

marg = full.groupby('highest_education', observed=True)['cate'].mean()
r_marg = marg[LTA] / marg[PG]

eff = full[full['code_module'] != 'CCC']
cell = (eff.groupby(['code_module', 'highest_education'], observed=True)['cate']
        .agg(['mean', 'count']).reset_index())
cell['_wm'] = cell['mean'] * cell['count']
grad = (cell.groupby('highest_education', observed=True)
        .agg(_num=('_wm', 'sum'), n=('count', 'sum')))
grad['n_weighted_cate'] = grad['_num'] / grad['n']
r_within = grad.loc[LTA, 'n_weighted_cate'] / grad.loc[PG, 'n_weighted_cate']

print(f"  marginal      LTA {marg[LTA]!r}  PG {marg[PG]!r}")
print(f"                ratio = {r_within!r}   <-- REPORT THIS")
print(f"\n  inflation factor = {r_marg / r_within:.4f}")
assert abs(r_marg - 1.82) < 0.01, f"marginal ratio drift: {r_marg!r}"
print("\n  The inflation is CCC holding 36.6% of all postgraduates at a CATE of")
print("  0.000129. Pooling across modules drags the postgraduate mean down for a")

four = [e for e in EDU_ORDER if e != 'No Formal quals']
seq = grad.reindex(four)['n_weighted_cate']
print(seq.to_string(float_format=lambda v: f"{v:.9f}"))
mono = bool((seq.diff().dropna() < 0).all())
print(f"\nmonotonic decreasing across the four: {mono}")
assert mono, "gradient is no longer monotonic -- investigate before reporting"


ALL MODULES
                         mean  count
code_presentation                   
2013B             0.005421672   2567
2013J             0.005434372   4939
2014B             0.004596298   3811
2014J             0.004125557   6212
  n-weighted   2014 - 2013 = np.float64(-0.0011254838192552036)

CCC REMOVED
                         mean  count
code_presentation                   
2013B             0.005421672   2567
2013J             0.005434372   4939
2014B             0.005907604   2943
2014J             0.005131823   4966
  n-weighted   2014 - 2013 = np.float64(-9.531124843251311e-06)

------------------------------------------------------------------------------
reproduced   : np.float64(0.0010670942262719692) with CCC, np.float64(-9.16918686948362e-05) without
both match the recorded values to 1e-6.

the gap collapses to 8.6% of its original size once CCC is
removed, and reverses sign -- there is no year trend, only module composition.
Without CCC, 2014B is the HIGHEST presenta

In [4]:
per_student = pd.DataFrame(shap_explanation.values, columns=X_shap_sample.columns)
per_student.insert(0, 'id_student',  full.loc[X_shap_sample.index, 'id_student'].values)
per_student.insert(1, 'code_module', full.loc[X_shap_sample.index, 'code_module'].values)
per_student.insert(2, 'cate',        full.loc[X_shap_sample.index, 'cate'].values)
per_student.to_csv(f'{BASE}/shap_per_student_500.csv', index=False)